In [4]:
import io, os, re, sys, warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
import ipywidgets as widgets

from scipy.signal import lfilter
from IPython.display import display, clear_output

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import TCN, TransformerMoment

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

## 1 · Configuration

In [ ]:
OPENSIM_ROOT = Path("/home/metamobility3/Jinwoo/AB03_Ilseung")
NPZ_ROOT = PROJECT_ROOT

CHECKPOINT_HIP = PROJECT_ROOT / "runs/0510_ik_id_hip_no_stair_epic_only/best_model.pt"
CHECKPOINT_KNEE = PROJECT_ROOT / "runs/0510_ik_id_knee_no_stair_epic_only/best_model.pt"

HIP_CFG_PATH = PROJECT_ROOT / "hip-exo-ctrl-V2/cfg/final.yaml"
KNEE_CFG_PATH = PROJECT_ROOT / "knee-exo-ctrl/cfg/final.yaml"

SUBJECT_MASS_KG = 84.4
MOCAP_FS = 1000.0
ID_LPF_CUTOFF_HZ = 6.0
ID_LPF_ORDER = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_LABELS = [
    ("hip_flexion_r",  "Hip Flexion R"),
    ("knee_angle_r",   "Knee Angle R"),
    ("ankle_angle_r",  "Ankle Angle R"),
    ("hip_flexion_l",  "Hip Flexion L"),
    ("knee_angle_l",   "Knee Angle L"),
    ("ankle_angle_l",  "Ankle Angle L"),
]
N_OUT = len(OUTPUT_LABELS)

ID_COLS = [
    "hip_flexion_r_moment",
    "knee_angle_r_moment",
    "ankle_angle_r_moment",
    "hip_flexion_l_moment",
    "knee_angle_l_moment",
    "ankle_angle_l_moment",
]

EXO_TORQUE_SCALE = {
    "hip-exo": 0.12,
    "knee-exo": 0.09,
}
EXO_SPEC = {
    "hip-exo": [("hip-exo", (0, 3))],
    "knee-exo": [("knee-exo", (1, 4))],
}

TRIAL_STEM_RE = re.compile(
    r"^(?P<subject>ab\d+_[^_]+)_(?P<exo>hip|knee)_(?P<speed>\dp\dmps)_(?P<cond>[a-z]{2})_exo_on$",
    re.IGNORECASE,
)

TRIALS = {}
for p in sorted(NPZ_ROOT.glob("*.npz")):
    m = TRIAL_STEM_RE.match(p.stem)
    if not m:
        continue
    g = m.groupdict()
    exo_folder = f"{g['exo'].lower()}-exo"
    cond = f"{g['cond'].upper()}_{g['speed']}"
    TRIALS[p.stem] = (exo_folder, cond, EXO_SPEC[exo_folder])

print("Trial manifest:")
for stem, (exo, cond, exo_spec) in TRIALS.items():
    ok = all([
        (NPZ_ROOT / f"{stem}.npz").exists(),
        (OPENSIM_ROOT / exo / "id" / f"{cond}_id.sto").exists(),
        (OPENSIM_ROOT / exo / "mocap" / f"{cond}.csv").exists(),
    ])
    print(f"  {'OK' if ok else 'MISS'}  {stem}  [{exo}]")

print()
print(f"Checkpoint hip : {'OK' if CHECKPOINT_HIP.exists() else 'MISS'}  {CHECKPOINT_HIP}")
print(f"Checkpoint knee: {'OK' if CHECKPOINT_KNEE.exists() else 'MISS'}  {CHECKPOINT_KNEE}")
print(f"Device: {DEVICE}")

Trial manifest:
  OK  ab01_jinwoo_hip_0p8mps_lg_exo_on  [hip-exo]
  OK  ab01_jinwoo_hip_0p8mps_ra_exo_on  [hip-exo]
  OK  ab01_jinwoo_hip_1p2mps_lg_exo_on  [hip-exo]
  OK  ab01_jinwoo_hip_1p6mps_lg_exo_on  [hip-exo]
  OK  ab01_jinwoo_knee_0p8mps_ra_exo_on  [knee-exo]
  OK  ab01_jinwoo_knee_0p8mps_rd_exo_on  [knee-exo]
  OK  ab02_oscar_hip_0p8mps_lg_exo_on  [hip-exo]
  OK  ab02_oscar_hip_0p8mps_ra_exo_on  [hip-exo]
  OK  ab02_oscar_hip_1p2mps_lg_exo_on  [hip-exo]
  OK  ab02_oscar_hip_1p6mps_lg_exo_on  [hip-exo]
  OK  ab02_oscar_knee_0p8mps_ra_exo_on  [knee-exo]
  OK  ab02_oscar_knee_0p8mps_rd_exo_on  [knee-exo]
  OK  ab03_ilseung_hip_0p8mps_lg_exo_on  [hip-exo]
  OK  ab03_ilseung_hip_0p8mps_ra_exo_on  [hip-exo]
  OK  ab03_ilseung_hip_1p2mps_lg_exo_on  [hip-exo]
  OK  ab03_ilseung_hip_1p6mps_lg_exo_on  [hip-exo]
  OK  ab03_ilseung_knee_0p8mps_ra_exo_on  [knee-exo]
  OK  ab03_ilseung_knee_0p8mps_rd_exo_on  [knee-exo]
  OK  ab04_changseob_hip_0p8mps_lg_exo_on  [hip-exo]
  OK  ab04_changseo

## 2 · Helper functions

In [6]:
@dataclass
class LoadedModel:
    model: torch.nn.Module
    window_size: int
    n_in: int
    n_out: int


def parse_opensim_file(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as fh:
        header_end = None
        for i, line in enumerate(fh):
            if line.strip().lower() == "endheader":
                header_end = i
                break
    if header_end is None:
        raise ValueError(f"Could not find endheader in {path}")
    df = pd.read_csv(path, sep=r"\s+", skiprows=header_end + 1)
    return df.set_index("time")


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS) -> tuple:
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines="skip")
    df = df[pd.to_numeric(df["Frame"], errors="coerce").notna()].copy()
    df["jet"] = pd.to_numeric(df["jet"], errors="coerce")
    df = df.dropna(subset=["jet"]).reset_index(drop=True)
    time = np.arange(len(df)) / fs
    return time, df["jet"].to_numpy(dtype=float)


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> int:
    above = np.asarray(signal, dtype=float) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def causal_lpf(signal: np.ndarray, fs: float, cutoff_hz: float, order: int) -> np.ndarray:
    if cutoff_hz <= 0:
        return np.asarray(signal, dtype=float)
    dt = 1.0 / float(fs)
    tau = 1.0 / (2.0 * np.pi * float(cutoff_hz))
    alpha = dt / (tau + dt)
    b = np.array([alpha], dtype=float)
    a = np.array([1.0, -(1.0 - alpha)], dtype=float)

    y = np.asarray(signal, dtype=float).copy()
    for _ in range(int(order)):
        zi = np.array([y[0] * (1.0 - alpha)], dtype=float)
        y, _ = lfilter(b, a, y, zi=zi)
    return y


def lpf_multichannel(x: np.ndarray, fs: float, cutoff_hz: float, order: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    for c in range(x.shape[1]):
        out[:, c] = causal_lpf(x[:, c], fs, cutoff_hz, order)
    return out


def estimate_fs(t: np.ndarray, default_fs: float = 100.0) -> float:
    t = np.asarray(t, dtype=float)
    if len(t) < 2:
        return default_fs
    dt = float(np.median(np.diff(t)))
    if dt <= 0 or not np.isfinite(dt):
        return default_fs
    return 1.0 / dt


def load_model_bundle(ckpt_path: Path) -> LoadedModel:
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=False)
    cfg = ckpt["model_config"]
    model_type = str(cfg.get("model_type", "tcn")).lower()

    if model_type == "tcn":
        model = TCN(
            n_input_channels=cfg["n_input_channels"],
            n_output_channels=cfg["n_output_channels"],
            hidden_channels=cfg["hidden_channels"],
            n_blocks=cfg["n_blocks"],
            kernel_size=cfg["kernel_size"],
            dropout=cfg["dropout"],
        )
    elif model_type == "transformer":
        model = TransformerMoment(
            n_input_channels=cfg["n_input_channels"],
            n_output_channels=cfg["n_output_channels"],
            d_model=cfg["d_model"],
            n_heads=cfg["n_heads"],
            n_layers=cfg["n_layers"],
            d_ff=cfg["d_ff"],
            dropout=cfg["dropout"],
        )
    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return LoadedModel(model=model, window_size=int(ckpt["window_size"]), n_in=int(cfg["n_input_channels"]), n_out=int(cfg["n_output_channels"]))


@torch.no_grad()
def _forward(model: torch.nn.Module, x_batch: np.ndarray) -> np.ndarray:
    xb = torch.from_numpy(np.ascontiguousarray(x_batch).astype(np.float32)).to(DEVICE)
    y = model(xb).detach().cpu().numpy()
    return np.transpose(y, (0, 2, 1))


def _causal_windows_single(model: torch.nn.Module, x: np.ndarray, W: int) -> np.ndarray:
    T, _ = x.shape
    p0 = _forward(model, x[:W].T[None, ...])[0]
    out = np.zeros((T, p0.shape[1]), dtype=np.float32)
    out[:min(T, W)] = p0[:min(T, W)]
    for s in range(1, T - W + 1):
        out[s + W - 1] = _forward(model, x[s:s+W].T[None, ...])[0, W - 1]
    return out


def _causal_windows_batched_rl(model: torch.nn.Module, x_r: np.ndarray, x_l: np.ndarray, W: int) -> tuple[np.ndarray, np.ndarray]:
    T, _ = x_r.shape
    x0 = np.stack([x_r[:W].T, x_l[:W].T], axis=0)
    p0 = _forward(model, x0)
    out_r = np.zeros((T, p0.shape[2]), dtype=np.float32)
    out_l = np.zeros((T, p0.shape[2]), dtype=np.float32)
    out_r[:min(T, W)] = p0[0, :min(T, W)]
    out_l[:min(T, W)] = p0[1, :min(T, W)]
    for s in range(1, T - W + 1):
        xb = np.stack([x_r[s:s+W].T, x_l[s:s+W].T], axis=0)
        pw = _forward(model, xb)
        out_r[s + W - 1] = pw[0, W - 1]
        out_l[s + W - 1] = pw[1, W - 1]
    return out_r, out_l


def exo_torque_from_pred(pred_nm_aligned: np.ndarray, exo_spec) -> np.ndarray:
    tau = np.zeros_like(pred_nm_aligned, dtype=float)
    for scale_key, channels in exo_spec:
        sc = float(EXO_TORQUE_SCALE[scale_key])
        for c in channels:
            tau[:, int(c)] += pred_nm_aligned[:, int(c)] * sc
    return tau


print("Helpers defined.")

Helpers defined.


## 3 · Load model checkpoints

In [7]:
HIP_CFG = yaml.safe_load(HIP_CFG_PATH.read_text())
KNEE_CFG = yaml.safe_load(KNEE_CFG_PATH.read_text())

HIP_BUNDLE = load_model_bundle(CHECKPOINT_HIP)
KNEE_BUNDLE = load_model_bundle(CHECKPOINT_KNEE)

print("Hip model:")
print(f"  in={HIP_BUNDLE.n_in}, out={HIP_BUNDLE.n_out}, window={HIP_BUNDLE.window_size}")
print("Knee model:")
print(f"  in={KNEE_BUNDLE.n_in}, out={KNEE_BUNDLE.n_out}, window={KNEE_BUNDLE.window_size}")

Hip model:
  in=2, out=1, window=100
Knee model:
  in=2, out=1, window=100


## 4 · Inference functions

In [8]:
def hip_inputs_from_npz(npz: np.lib.npyio.NpzFile, cfg: dict) -> tuple[np.ndarray, np.ndarray, float]:
    t = np.asarray(npz["time"], dtype=float)
    fs = estimate_fs(t, default_fs=float(cfg.get("fs", 100.0)))

    angle_r_raw = np.asarray(npz["hip_pos_R"], dtype=float)
    angle_l_raw = np.asarray(npz["hip_pos_L"], dtype=float)
    vel_r_raw = -np.deg2rad(np.asarray(npz["imu_R"], dtype=float)[:, 4])
    vel_l_raw = -np.deg2rad(np.asarray(npz["imu_L"], dtype=float)[:, 4])

    angle_hz = float(cfg.get("angle_lpf_hz", 6.0))
    angle_order = int(cfg.get("angle_lpf_order", 4))
    vel_hz = float(cfg.get("vel_lpf_hz", 15.0))
    vel_order = int(cfg.get("vel_lpf_order", 4))

    angle_r = causal_lpf(angle_r_raw, fs, angle_hz, angle_order)
    angle_l = causal_lpf(angle_l_raw, fs, angle_hz, angle_order)
    vel_r = causal_lpf(vel_r_raw, fs, vel_hz, vel_order)
    vel_l = causal_lpf(vel_l_raw, fs, vel_hz, vel_order)

    x_r = np.stack([angle_r, vel_r], axis=1).astype(np.float32)
    x_l = np.stack([angle_l, vel_l], axis=1).astype(np.float32)
    return x_r, x_l, fs


def knee_inputs_from_npz(npz: np.lib.npyio.NpzFile, cfg: dict) -> tuple[np.ndarray, float]:
    t = np.asarray(npz["time"], dtype=float)
    fs = estimate_fs(t, default_fs=float(cfg.get("fs", 100.0)))
    angle = np.asarray(npz["model_in_knee_angle_raw"], dtype=float)
    vel = np.asarray(npz["model_in_knee_vel_raw"], dtype=float)
    x = np.stack([angle, vel], axis=1).astype(np.float32)
    return x, fs


def run_inference_from_npz(stem: str, npz: np.lib.npyio.NpzFile, exo_folder: str) -> np.ndarray:
    T = len(npz["time"])
    pred_nmpkg = np.full((T, N_OUT), np.nan, dtype=np.float32)

    if exo_folder == "hip-exo":
        x_r, x_l, fs = hip_inputs_from_npz(npz, HIP_CFG)
        pr, pl = _causal_windows_batched_rl(HIP_BUNDLE.model, x_r, x_l, HIP_BUNDLE.window_size)

        out_hz = float(HIP_CFG.get("out_lpf_hz", 6.0))
        out_order = int(HIP_CFG.get("out_lpf_order", 4))
        pr = causal_lpf(pr[:, 0], fs, out_hz, out_order)
        pl = causal_lpf(pl[:, 0], fs, out_hz, out_order)

        pred_nmpkg[:, 0] = pr
        pred_nmpkg[:, 3] = pl

    elif exo_folder == "knee-exo":
        x, fs = knee_inputs_from_npz(npz, KNEE_CFG)
        p = _causal_windows_single(KNEE_BUNDLE.model, x, KNEE_BUNDLE.window_size)[:, 0]

        out_hz = float(KNEE_CFG.get("out_lpf_hz", 6.0))
        out_order = int(KNEE_CFG.get("out_lpf_order", 4))
        p = causal_lpf(p, fs, out_hz, out_order)

        cmd_r = np.asarray(npz["cmd_R"], dtype=float) if "cmd_R" in npz.files else np.zeros(T)
        cmd_l = np.asarray(npz["cmd_L"], dtype=float) if "cmd_L" in npz.files else np.zeros(T)
        if np.nanmean(np.abs(cmd_l)) > np.nanmean(np.abs(cmd_r)):
            pred_nmpkg[:, 4] = p
        else:
            pred_nmpkg[:, 1] = p
    else:
        raise ValueError(f"Unknown exo folder: {exo_folder}")

    return pred_nmpkg * float(SUBJECT_MASS_KG)


print("Inference functions defined.")

Inference functions defined.


## 5 · Pre-load all trials

In [9]:
TRIAL_DATA = {}

for stem, (exo_folder, cond, exo_spec) in TRIALS.items():
    npz_path = NPZ_ROOT / f"{stem}.npz"
    id_path = OPENSIM_ROOT / exo_folder / "id" / f"{cond}_id.sto"
    mocap_path = OPENSIM_ROOT / exo_folder / "mocap" / f"{cond}.csv"

    if not all(p.exists() for p in (npz_path, id_path, mocap_path)):
        print(f"[SKIP] {stem} - missing files")
        continue

    print(f"Processing {stem} ...", end=" ", flush=True)

    npz = np.load(npz_path, allow_pickle=True)
    t_npz = np.asarray(npz["time"], dtype=float)
    gpio_npz = np.asarray(npz["GPIO"], dtype=float)

    pred_nm = run_inference_from_npz(stem, npz, exo_folder)

    t_mocap, gpio_mocap = parse_mocap_csv(mocap_path)
    g_rng = gpio_mocap.max() - gpio_mocap.min()
    gpio_mocap_norm = (gpio_mocap - gpio_mocap.min()) / g_rng if g_rng > 0 else gpio_mocap

    idx_npz = first_falling_edge(gpio_npz)
    idx_mocap = first_falling_edge(gpio_mocap_norm)
    if idx_npz is None or idx_mocap is None:
        print("[no GPIO edge]")
        continue

    t_offset = float(t_mocap[idx_mocap] - t_npz[idx_npz])
    t_npz_aligned = t_npz + t_offset

    delay_s = float((HIP_CFG if exo_folder == "hip-exo" else KNEE_CFG).get("delay", 0.0))
    t_id_eval = t_npz_aligned - delay_s

    id_df = parse_opensim_file(id_path)
    t_id = id_df.index.to_numpy(float)
    id_fs = estimate_fs(t_id, default_fs=100.0)

    id_raw = np.zeros((len(t_id), N_OUT), dtype=np.float64)
    for c, col in enumerate(ID_COLS):
        id_raw[:, c] = id_df[col].to_numpy(float) if col in id_df.columns else np.nan
    id_filt = lpf_multichannel(id_raw, fs=id_fs, cutoff_hz=ID_LPF_CUTOFF_HZ, order=ID_LPF_ORDER)

    mask = (t_id_eval >= t_id[0]) & (t_id_eval <= t_id[-1])
    if not np.any(mask):
        print("[no overlap]")
        continue

    t_common = t_npz_aligned[mask]
    pred_common = pred_nm[mask]

    id_interp = np.stack([
        np.interp(t_id_eval[mask], t_id, id_filt[:, c], left=np.nan, right=np.nan)
        for c in range(N_OUT)
    ], axis=1)

    exo_torque = exo_torque_from_pred(pred_common, exo_spec)
    id_net = id_interp - exo_torque

    metrics = []
    for c in range(N_OUT):
        v = np.isfinite(pred_common[:, c]) & np.isfinite(id_net[:, c])
        if v.sum() > 1:
            rmse = float(np.sqrt(np.mean((pred_common[v, c] - id_net[v, c]) ** 2)))
            r2 = float(np.corrcoef(pred_common[v, c], id_net[v, c])[0, 1]) ** 2
        else:
            rmse, r2 = np.nan, np.nan
        metrics.append({"rmse": rmse, "r2": r2})

    TRIAL_DATA[stem] = {
        "stem": stem,
        "exo": exo_folder,
        "cond": cond,
        "exo_spec": exo_spec,
        "t": t_common,
        "t_rel": t_common - t_common[0],
        "pred_nm": pred_common,
        "id_nm_gross": id_interp,
        "exo_torque_nm": exo_torque,
        "id_nm": id_net,
        "metrics": metrics,
        "t_offset": t_offset,
        "delay_s": delay_s,
    }

    mean_r2 = float(np.nanmean([m["r2"] for m in metrics]))
    mean_rmse = float(np.nanmean([m["rmse"] for m in metrics]))
    dur = float(t_common[-1] - t_common[0])
    print(f"ok  ({dur:.1f} s, mean R2={mean_r2:.3f}, mean RMSE={mean_rmse:.2f} N*m)")

print(f"\nLoaded {len(TRIAL_DATA)} / {len(TRIALS)} trials.")

Processing ab01_jinwoo_hip_0p8mps_lg_exo_on ... ok  (79.6 s, mean R2=0.026, mean RMSE=32.95 N*m)
Processing ab01_jinwoo_hip_0p8mps_ra_exo_on ... ok  (79.7 s, mean R2=0.013, mean RMSE=48.63 N*m)
Processing ab01_jinwoo_hip_1p2mps_lg_exo_on ... ok  (78.5 s, mean R2=0.017, mean RMSE=40.48 N*m)
Processing ab01_jinwoo_hip_1p6mps_lg_exo_on ... ok  (77.6 s, mean R2=0.011, mean RMSE=48.29 N*m)
Processing ab01_jinwoo_knee_0p8mps_ra_exo_on ... ok  (79.8 s, mean R2=0.007, mean RMSE=37.07 N*m)
Processing ab01_jinwoo_knee_0p8mps_rd_exo_on ... ok  (79.8 s, mean R2=0.004, mean RMSE=36.19 N*m)
Processing ab02_oscar_hip_0p8mps_lg_exo_on ... ok  (79.6 s, mean R2=0.123, mean RMSE=31.36 N*m)
Processing ab02_oscar_hip_0p8mps_ra_exo_on ... ok  (79.7 s, mean R2=0.022, mean RMSE=51.40 N*m)
Processing ab02_oscar_hip_1p2mps_lg_exo_on ... ok  (79.6 s, mean R2=0.001, mean RMSE=36.15 N*m)
Processing ab02_oscar_hip_1p6mps_lg_exo_on ... ok  (79.6 s, mean R2=0.267, mean RMSE=31.32 N*m)
Processing ab02_oscar_knee_0p8mp

In [10]:
## 5b · Inspect model inputs (interactive)
_INPUT_CACHE = {}

def _load_input_data(stem: str) -> dict:
    if stem in _INPUT_CACHE:
        return _INPUT_CACHE[stem]
    exo = TRIALS[stem][0]
    npz = np.load(NPZ_ROOT / f"{stem}.npz", allow_pickle=True)
    t = np.asarray(npz["time"], dtype=float)
    if exo == "hip-exo":
        xr, xl, fs = hip_inputs_from_npz(npz, HIP_CFG)
        d = dict(t=t, t_rel=t - t[0], mode="hip", xr=xr, xl=xl, fs=fs)
    else:
        x, fs = knee_inputs_from_npz(npz, KNEE_CFG)
        d = dict(t=t, t_rel=t - t[0], mode="knee", x=x, fs=fs)
    _INPUT_CACHE[stem] = d
    return d

_first_stem_inp = next(iter(TRIAL_DATA))
_first_inp = _load_input_data(_first_stem_inp)

inp_trial_dropdown = widgets.Dropdown(options=list(TRIAL_DATA.keys()), value=_first_stem_inp, description="Trial:", layout=widgets.Layout(width="500px"), style={"description_width": "80px"})
inp_time_slider = widgets.FloatRangeSlider(value=[float(_first_inp["t_rel"][0]), float(_first_inp["t_rel"][-1])], min=float(_first_inp["t_rel"][0]), max=float(_first_inp["t_rel"][-1]), step=0.5, description="Time (s):", layout=widgets.Layout(width="700px"), style={"description_width": "80px"}, continuous_update=False, readout_format=".1f")
inp_out = widgets.Output()

def _draw_input(stem: str, t0: float, t1: float):
    d = _load_input_data(stem)
    m = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t = d["t_rel"][m]
    if d["mode"] == "hip":
        fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        ax[0].plot(t, d["xr"][m, 0], label="angle R")
        ax[0].plot(t, d["xl"][m, 0], label="angle L")
        ax[0].set_ylabel("rad")
        ax[0].set_title(f"{stem} | hip input angle")
        ax[0].grid(alpha=0.3)
        ax[0].legend()
        ax[1].plot(t, d["xr"][m, 1], label="vel R")
        ax[1].plot(t, d["xl"][m, 1], label="vel L")
        ax[1].set_ylabel("rad/s")
        ax[1].set_xlabel("Time [s]")
        ax[1].set_title(f"{stem} | hip input velocity")
        ax[1].grid(alpha=0.3)
        ax[1].legend()
    else:
        fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        ax[0].plot(t, d["x"][m, 0], label="angle")
        ax[0].set_ylabel("rad")
        ax[0].set_title(f"{stem} | knee input angle")
        ax[0].grid(alpha=0.3)
        ax[0].legend()
        ax[1].plot(t, d["x"][m, 1], label="velocity")
        ax[1].set_ylabel("rad/s")
        ax[1].set_xlabel("Time [s]")
        ax[1].set_title(f"{stem} | knee input velocity")
        ax[1].grid(alpha=0.3)
        ax[1].legend()
    plt.tight_layout()
    plt.show()


def _update_input_slider(change):
    d = _load_input_data(inp_trial_dropdown.value)
    inp_time_slider.min = float(d["t_rel"][0])
    inp_time_slider.max = float(d["t_rel"][-1])
    inp_time_slider.value = (float(d["t_rel"][0]), float(d["t_rel"][-1]))


def _refresh_input(*_):
    with inp_out:
        clear_output(wait=True)
        t0, t1 = inp_time_slider.value
        _draw_input(inp_trial_dropdown.value, t0, t1)

inp_trial_dropdown.observe(_update_input_slider, names="value")
inp_trial_dropdown.observe(_refresh_input, names="value")
inp_time_slider.observe(_refresh_input, names="value")

display(inp_trial_dropdown)
display(inp_time_slider)
display(inp_out)
_refresh_input()

Dropdown(description='Trial:', layout=Layout(width='500px'), options=('ab01_jinwoo_hip_0p8mps_lg_exo_on', 'ab0…

FloatRangeSlider(value=(0.0, 79.98993651599994), continuous_update=False, description='Time (s):', layout=Layo…

Output()

## 6 · Interactive comparison

In [ ]:
_first_stem = next(iter(TRIAL_DATA))
_first_d = TRIAL_DATA[_first_stem]

trial_dropdown = widgets.Dropdown(options=list(TRIAL_DATA.keys()), value=_first_stem, description="Trial:", layout=widgets.Layout(width="500px"), style={"description_width": "80px"})
time_slider = widgets.FloatRangeSlider(value=[float(_first_d["t_rel"][0]), float(_first_d["t_rel"][-1])], min=float(_first_d["t_rel"][0]), max=float(_first_d["t_rel"][-1]), step=0.5, description="Time (s):", layout=widgets.Layout(width="700px"), style={"description_width": "80px"}, continuous_update=False, readout_format=".1f")
show_residual = widgets.ToggleButton(value=False, description="Show residuals", button_style="", icon="signal", layout=widgets.Layout(width="160px"))
out = widgets.Output()


def _draw(stem: str, t0: float, t1: float, residuals: bool) -> None:
    d = TRIAL_DATA[stem]
    mask = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t = d["t_rel"][mask]
    pred = d["pred_nm"][mask]
    id_m = d["id_nm"][mask]

    n_rows = N_OUT * 2 if residuals else N_OUT
    fig_h = n_rows * 2.2 + 0.6
    fig, axes = plt.subplots(n_rows, 1, figsize=(15, fig_h), sharex=True)
    if n_rows == 1:
        axes = [axes]

    row = 0
    for c_idx, (_key, label) in enumerate(OUTPUT_LABELS):
        ax = axes[row]
        ax.plot(t, id_m[:, c_idx], color="#1E88E5", lw=1.5, label="OpenSim ID - tau_exo")
        ax.plot(t, pred[:, c_idx], color="#E53935", lw=1.3, label="Model replay")
        m = d["metrics"][c_idx]
        ax.set_title(f"{label}   RMSE={m['rmse']:.2f} N*m   R2={m['r2']:.3f}")
        ax.set_ylabel("N*m")
        ax.grid(alpha=0.3)
        ax.legend(loc="upper right", fontsize=8)
        row += 1

        if residuals:
            ar = axes[row]
            ar.plot(t, pred[:, c_idx] - id_m[:, c_idx], color="#8E24AA", lw=1.1)
            ar.axhline(0.0, color="k", ls="--", lw=1.0, alpha=0.6)
            ar.set_ylabel("Err")
            ar.grid(alpha=0.3)
            row += 1

    mean_r2 = float(np.nanmean([d["metrics"][c]["r2"] for c in range(N_OUT)]))
    mean_rmse = float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)]))
    fig.suptitle(f"{stem}   Mean R2={mean_r2:.3f}   Mean RMSE={mean_rmse:.2f} N*m", y=1.002)
    axes[-1].set_xlabel("Time [s]")
    plt.tight_layout()
    plt.show()


def _on_trial_change(change):
    d = TRIAL_DATA[change["new"]]
    time_slider.min = float(d["t_rel"][0])
    time_slider.max = float(d["t_rel"][-1])
    time_slider.value = (float(d["t_rel"][0]), float(d["t_rel"][-1]))


def _refresh(*_):
    with out:
        clear_output(wait=True)
        t0, t1 = time_slider.value
        _draw(trial_dropdown.value, t0, t1, show_residual.value)

trial_dropdown.observe(_on_trial_change, names="value")
trial_dropdown.observe(_refresh, names="value")
time_slider.observe(_refresh, names="value")
show_residual.observe(_refresh, names="value")

display(widgets.HBox([trial_dropdown, show_residual]))
display(time_slider)
display(out)
_refresh()

FloatRangeSlider(value=(0.0, 79.64998110199986), continuous_update=False, description='Time (s):', layout=Layo…

Output()

## 7 · Window statistics

In [12]:
t0_s, t1_s = time_slider.value
stem_s = trial_dropdown.value
d_s = TRIAL_DATA[stem_s]

mask_s = (d_s["t_rel"] >= t0_s) & (d_s["t_rel"] <= t1_s)
pred_s = d_s["pred_nm"][mask_s]
id_s = d_s["id_nm"][mask_s]

print(f"Trial  : {stem_s}")
print(f"Window : {t0_s:.1f} - {t1_s:.1f} s  ({t1_s - t0_s:.1f} s)")
print()
print(f"{'Channel':<30} {'RMSE [N*m]':>12} {'R2':>8} {'peak model':>11} {'peak ID net':>12}")
print("-" * 84)

mean_r2_vals, mean_rmse_vals = [], []
for c, (_key, label) in enumerate(OUTPUT_LABELS):
    v = np.isfinite(pred_s[:, c]) & np.isfinite(id_s[:, c])
    if v.sum() > 1:
        rmse_c = float(np.sqrt(np.mean((pred_s[v, c] - id_s[v, c]) ** 2)))
        r2_c = float(np.corrcoef(pred_s[v, c], id_s[v, c])[0, 1]) ** 2
        pk_m = float(np.nanmax(np.abs(pred_s[v, c])))
        pk_i = float(np.nanmax(np.abs(id_s[v, c])))
        mean_r2_vals.append(r2_c)
        mean_rmse_vals.append(rmse_c)
    else:
        rmse_c = r2_c = pk_m = pk_i = np.nan

    print(f"  {label:<28} {rmse_c:>12.3f} {r2_c:>8.3f} {pk_m:>11.2f} {pk_i:>12.2f}")

print()
print(f"  {'Mean (all channels)':<28} {np.nanmean(mean_rmse_vals):>12.3f} {np.nanmean(mean_r2_vals):>8.3f}")

Trial  : ab01_jinwoo_hip_0p8mps_lg_exo_on
Window : 0.0 - 79.6 s  (79.6 s)

Channel                          RMSE [N*m]       R2  peak model  peak ID net
------------------------------------------------------------------------------------
  Hip Flexion R                      33.111    0.025       48.41        42.11
  Knee Angle R                          nan      nan         nan          nan
  Ankle Angle R                         nan      nan         nan          nan
  Hip Flexion L                      32.783    0.027       45.17        39.79
  Knee Angle L                          nan      nan         nan          nan
  Ankle Angle L                         nan      nan         nan          nan

  Mean (all channels)                32.947    0.026


## 8 · All-trials summary (full overlap)

In [13]:
rows = []
for stem, d in TRIAL_DATA.items():
    row = {
        "trial": stem,
        "exo": d["exo"],
        "condition": d["cond"],
        "ID LPF": f"{ID_LPF_CUTOFF_HZ:.0f} Hz {ID_LPF_ORDER}-ord causal",
        "GT ID": "ID - tau_exo",
        "overlap [s]": round(float(d["t_rel"][-1] - d["t_rel"][0]), 1),
        "t_offset [s]": round(float(d["t_offset"]), 4),
        "delay [s]": round(float(d["delay_s"]), 3),
    }

    for c, (key, _label) in enumerate(OUTPUT_LABELS):
        m = d["metrics"][c]
        row[f"RMSE_{key} [N*m]"] = round(m["rmse"], 2) if np.isfinite(m["rmse"]) else np.nan
        row[f"R2_{key}"] = round(m["r2"], 3) if np.isfinite(m["r2"]) else np.nan

    row["mean_RMSE [N*m]"] = round(float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)])), 2)
    row["mean_R2"] = round(float(np.nanmean([d["metrics"][c]["r2"] for c in range(N_OUT)])), 3)
    rows.append(row)

pd.DataFrame(rows).sort_values(["exo", "trial"]).reset_index(drop=True)

,trial,exo,condition,ID LPF,GT ID,overlap [s],t_offset [s],delay [s],RMSE_hip_flexion_r [N*m],R2_hip_flexion_r,...,RMSE_ankle_angle_r [N*m],R2_ankle_angle_r,RMSE_hip_flexion_l [N*m],R2_hip_flexion_l,RMSE_knee_angle_l [N*m],R2_knee_angle_l,RMSE_ankle_angle_l [N*m],R2_ankle_angle_l,mean_RMSE [N*m],mean_R2
0,ab01_jinwoo_hip_0p8mps_lg_exo_on,hip-exo,LG_0p8mps,6 Hz 4-ord causal,ID - tau_exo,79.6,-0.2251,0.12,33.11,0.025,...,NaN,NaN,32.78,0.027,NaN,NaN,NaN,NaN,32.95,0.026
1,ab01_jinwoo_hip_0p8mps_ra_exo_on,hip-exo,RA_0p8mps,6 Hz 4-ord causal,ID - tau_exo,79.7,-0.1511,0.12,50.18,0.014,...,NaN,NaN,47.08,0.012,NaN,NaN,NaN,NaN,48.63,0.013
2,ab01_jinwoo_hip_1p2mps_lg_exo_on,hip-exo,LG_1p2mps,6 Hz 4-ord causal,ID - tau_exo,78.5,-0.2260,0.12,41.17,0.017,...,NaN,NaN,39.78,0.017,NaN,NaN,NaN,NaN,40.48,0.017
3,ab01_jinwoo_hip_1p6mps_lg_exo_on,hip-exo,LG_1p6mps,6 Hz 4-ord causal,ID - tau_exo,77.6,-0.2360,0.12,48.90,0.011,...,NaN,NaN,47.68,0.010,NaN,NaN,NaN,NaN,48.29,0.011
4,ab02_oscar_hip_0p8mps_lg_exo_on,hip-exo,LG_0p8mps,6 Hz 4-ord causal,ID - tau_exo,79.6,-0.2350,0.12,30.79,0.115,...,NaN,NaN,31.92,0.131,NaN,NaN,NaN,NaN,31.36,0.123
5,ab02_oscar_hip_0p8mps_ra_exo_on,hip-exo,RA_0p8mps,6 Hz 4-ord causal,ID - tau_exo,79.7,-0.1410,0.12,51.12,0.017,...,NaN,NaN,51.68,0.028,NaN,NaN,NaN,NaN,51.40,0.022
6,ab02_oscar_hip_1p2mps_lg_exo_on,hip-exo,LG_1p2mps,6 Hz 4-ord causal,ID - tau_exo,79.6,-0.2261,0.12,34.62,0.002,...,NaN,NaN,37.69,0.000,NaN,NaN,NaN,NaN,36.15,0.001
7,ab02_oscar_hip_1p6mps_lg_exo_on,hip-exo,LG_1p6mps,6 Hz 4-ord causal,ID - tau_exo,79.6,-0.2360,0.12,30.86,0.270,...,NaN,NaN,31.77,0.265,NaN,NaN,NaN,NaN,31.32,0.267
8,ab03_ilseung_hip_0p8mps_lg_exo_on,hip-exo,LG_0p8mps,6 Hz 4-ord causal,ID - tau_exo,79.6,-0.2350,0.12,13.03,0.604,...,NaN,NaN,17.10,0.486,NaN,NaN,NaN,NaN,15.07,0.545
9,ab03_ilseung_hip_0p8mps_ra_exo_on,hip-exo,RA_0p8mps,6 Hz 4-ord causal,ID - tau_exo,79.7,-0.1411,0.12,22.45,0.596,...,NaN,NaN,22.13,0.623,NaN,NaN,NaN,NaN,22.29,0.610
